In [ ]:
<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_hotelchain_powerbi_guide.png" width="100%"/>
</div>




---

> **Objectif :** transformer les CSV analytiques du NB2 en un **dashboard Power BI 5 pages** opérationnel pour Marc-Aurèle, avec **75 mesures DAX**, une narrative CODIR en 5 slides (SCQRA), et un plan de déploiement chez les 5 clients de l'agence FluxData.

| | |
|---|---|
| **Livrable** | Dashboard Power BI 5 pages + 75 mesures DAX + plan déploiement |
| **Pattern data viz** | Étoile à 7 tables, mesures par dossier, layout multi-pages |
| **Durée estimée** | 4 à 6 heures |
| **Cible** | Marc-Aurèle (DPM agence) + 5 clients en self-service |
| **Outils** | Power BI Desktop, DAX, visuel HTML Content (AppSource) |


---
## Guide de lecture du notebook

Trois types de blocs pédagogiques rythment le notebook :

| Icône | Bloc | Rôle |
|---|---|---|
| 🎓 | **MÉTHODE** | Technique Power BI / DAX / modélisation, avec le **pourquoi** |
| 💡 | **INTERPRÉTATION** | Lecture des résultats : ce que les chiffres signifient |
| 🏢 | **MÉTIER** | Décision Paid Media à prendre à partir de l'analyse |

### Architecture du livrable final

```
📊 Dashboard GoogleAdsPulse (.pbix)
├── 01 Overview      ......... vue globale des 5 comptes · 24 mois
├── 02 Campaigns     ......... top/bottom 5 ROAS + tableau classement
├── 03 Keywords      ......... 3 KPI cards + cohort retention
├── 04 Conversions   ......... donut types + cost/value + top campagnes
└── 05 Breakdown     ......... devices + heatmap horaire + pays + types
```

### Étapes de construction (ordre conseillé)

1. Connexion Power BI aux CSV / tables analytiques
2. Modèle en étoile + table Calendrier
3. Création des 75 mesures DAX (par dossier)
4. Pages 01 à 05 dans l'ordre
5. Sous-titres dynamiques + navigation
6. Validation finale (SQL/Python vs Power BI)
7. Storytelling CODIR (SCQRA 5 slides)
8. Déploiement + RLS + refresh


---
## Contexte du livrable

Marc-Aurèle est Paid Media Manager chez **FluxData Agency**. Il pilote 5 comptes Google Ads pour des clients de secteurs variés (e-commerce TechShop, hôtellerie AfriHotels, banque BankAfrica, formation EdTechCI, médical MedSupply). Chaque lundi matin il passe 4 heures à compiler 5 rapports Excel pour ses clients. Ce dashboard remplace ce travail manuel.

### 🎯 Les 5 questions clients auxquelles le dashboard répond

| Page | Question business | Réponse cible |
|---|---|---|
| 01 — Overview | Comment se comportent mes 5 comptes ce mois-ci ? | 56M imp · 1,5M clics · €625K spend · ROAS 5,96× |
| 02 — Campaigns | Quelles campagnes performent et lesquelles drainent le budget ? | Top : Brand-MedSupply 19,2× · Bottom : Search-MedicalEquip 0,3× |
| 03 — Keywords | Quels mots-clés sont à pauser ? À booster ? | 23 à pauser · top CTR 17% · QS moyen 6,7 |
| 04 — Conversions | D'où viennent les conversions et combien coûtent-elles ? | 59,7K conversions · 63% Purchase · Lead = CPA le moins cher (€7,53) |
| 05 — Breakdown | Quels segments (device, horaire, pays) optimiser ? | Mobile 52% · pic 9-12h · Côte d'Ivoire 40% du spend |

### 🎨 Identité visuelle du dashboard

| Élément | Valeur |
|---|---|
| Police principale | Segoe UI |
| Couleur primaire (sidebar/active) | Violet `#7B61FF` |
| Couleur succès / vert | `#1D9E75` |
| Couleur alerte / rouge | `#E25F2F` |
| Couleur warning / orange | `#EF9F27` |
| Couleur info / bleu | `#1B9CFC` |
| Couleur neutre | `#7891B5` |
| Logo / branding | "GoogleAdsPulse · Paid Media Analytics" en haut-gauche |

> **🎓 MÉTHODE — Pourquoi un thème couleur fixe ?**
>
> Un dashboard exécutif doit avoir **une charte graphique cohérente** sur toutes les pages. Power BI permet de créer un fichier thème `.json` (Affichage → Thèmes → Personnaliser le thème). Les mêmes couleurs doivent toujours coder la même chose : violet = action principale, vert = OK, orange = vigilance, rouge = alerte. Cette règle de cohérence réduit la charge cognitive du lecteur de ~40% selon les études UX dataviz.


---
## Étape 1 — Connexion Power BI aux données

### 🎓 MÉTHODE — Quels objets charger ?

Power BI charge les **5 tables sources** (faits + dimensions) **+ 6 tables analytiques** générées au NB2 :

```
✅ Tables sources (transactionnelles)        ✅ Tables analytiques (NB2)
─────────────────────────────────────        ───────────────────────────────────
performance_quotidienne (table de faits)     gads_kpi_mensuel
campaigns                                    gads_anomalies
ads                                          gads_benchmark
keywords                                     gads_campaigns_rank
accounts                                     gads_cohort_keywords
                                             gads_jour_heure
                                             gads_rolling
```

### Procédure de connexion

```
Power BI Desktop
└── Accueil
    └── Obtenir des données
        ├── CSV (si fichiers) → pointer le dossier output du NB2
        └── ou SQL Server / DuckDB / Fabric (selon ton infra)
```

> **🎓 MÉTHODE — Mode Import vs DirectQuery**
>
> **Import** charge en mémoire VertiPaq (compresse ~10×). Recommandé pour GoogleAdsPulse (<200 Mo de données). Toutes les fonctions DAX sont disponibles.
>
> **DirectQuery** envoie chaque requête au serveur en temps réel. Réservé aux très gros volumes (>1 GB) ou besoin de fraîcheur seconde par seconde.


---
## Étape 2 — Modèle de données en étoile

### 🎓 MÉTHODE — Pourquoi le schéma en étoile ?

Power BI est optimisé pour le **schéma en étoile** : une table de faits centrale entourée de tables de dimension. Le moteur VertiPaq compresse et interroge ce schéma **10 à 100× plus vite** qu'un schéma normalisé.

```
                ┌──────────────┐
                │  accounts    │  (dim)
                └──────┬───────┘
                       │ account_id
   ┌────────────┐      │      ┌──────────────┐
   │ campaigns  │──────┼──────│  Calendrier  │
   │   (dim)    │      │      │   (dim)      │
   └─────┬──────┘      │      └──────┬───────┘
         │             │             │
         │  ┌──────────▼────────────────────┐  date
         └──│  performance_quotidienne      │──┘
            │           (FAIT)              │
            └───┬───────────────────┬───────┘
                │                   │
                │ keyword_id        │ ad_id
                │                   │
        ┌───────▼──────┐    ┌───────▼──────┐
        │  keywords    │    │     ads      │
        │    (dim)     │    │    (dim)     │
        └──────────────┘    └──────────────┘
```

### Relations à créer (Modélisation → Gérer les relations)

| Côté 1 (dim) | Côté * (fait) | Clé | Cardinalité |
|---|---|---|---|
| `accounts[account_id]` | `performance_quotidienne[account_id]` | account_id | 1 → * |
| `campaigns[campaign_id]` | `performance_quotidienne[campaign_id]` | campaign_id | 1 → * |
| `keywords[keyword_id]` | `performance_quotidienne[keyword_id]` | keyword_id | 1 → * |
| `ads[ad_id]` | `performance_quotidienne[ad_id]` | ad_id | 1 → * |
| `Calendrier[Date]` | `performance_quotidienne[date]` | Date | 1 → * |
| `accounts[account_id]` | `campaigns[account_id]` | account_id | 1 → * (relation inactive) |
| `campaigns[campaign_id]` | `keywords[campaign_id]` | campaign_id | 1 → * (relation inactive) |


In [ ]:
-- Formule DAX pour créer la table Calendrier
-- Accueil → Nouvelle table

dim_calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2023, 1, 1), DATE(2024, 12, 31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMM"),
    "Mois_Court",   FORMAT([Date], "MMM YYYY"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semestre",     IF(MONTH([Date]) <= 6, "S1", "S2"),
    "Jour_Semaine", FORMAT([Date], "dddd"),
    "Num_Jour",     WEEKDAY([Date], 2),
    "Est_Weekend",  IF(WEEKDAY([Date], 2) >= 6, "Weekend", "Semaine"),
    "Jour_Initial", LEFT(FORMAT([Date], "dddd"), 1)
)

-- Après création : Modélisation → Marquer comme table de dates → Date

---
## Étape 3 — Les 85 mesures DAX

### 🎓 MÉTHODE — Organiser les mesures par dossier d'affichage

Crée une **table vide nommée `_Mesures`** (Accueil → Entrer des données → Charger). Toutes les mesures vivent dans cette table, classées par **dossier d'affichage** (display folder) pour une navigation propre dans le panneau Champs.

### Vue d'ensemble — 85 mesures réparties en 11 dossiers

| Dossier | Nb | Mesures clés |
|---|---|---|
| **1. Volume** | 3 | Total Impressions · Total Clicks · Total Conversions |
| **2. Efficacité** | 5 | CTR · CPC · CPM · Cost per Conversion · Conversion Rate |
| **3. Rentabilité** | 3 | Total Spend · Conversions Value · ROAS |
| **4. Période précédente** | 10 | Impressions Prev · Clicks Prev · ... · Conversions Value Prev |
| **5. Deltas MoM** | 10 | Impressions Delta % · ... · Conversions Value Delta % |
| **6. Conversions par type** | 5 | Purchase · Lead · Signup · Demo · Purchase Value |
| **7. Contextuelles** | 9 | ROAS Verdict · Spend 7d MA · Spend YTD · Nb Campagnes Actives · QS Moyen · Cost per Purchase · **ROAS Verdict Icon** ★ · **Qrt** ★ |
| **8. Keywords KPI** | 4 | **Nb Keywords Actifs** ★ · **Keywords A Pauser** ★ · **Top CTR Keyword Nom** ★ · **Top CTR Keyword Valeur** ★ |
| **9. Devices** | 3 | **% Clicks Desktop** ★ · **% Clicks Mobile** ★ · **% Clicks Tablet** ★ |
| **10. HTML Visuels** | 1 | **Heatmap Jour Heure HTML** ★ |
| **_Helpers** | 10 | **Couleurs conditionnelles** pour KPI cards (vert/rouge selon delta) ★ |

★ = mesures de couleur ajoutées pour le pilotage UP-IS-GOOD vs DOWN-IS-GOOD des KPI cards Page 01.

★ = mesures spécifiques aux pages 02 / 03 / 05 (ajoutées dans cette version 2)


In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 1 — VOLUME (3 mesures)
-- ════════════════════════════════════════════════════════

Total Impressions = SUM(performance_quotidienne[impressions])
-- ✅ Sans filtre : 56 018 776 (~56,02M)

Total Clicks = SUM(performance_quotidienne[clicks])
-- ✅ Sans filtre : 1 488 092 (~1,49M)

Total Conversions = SUM(performance_quotidienne[conversions])
-- ✅ Sans filtre : 59 715 (~59,7K)

-- ════════════════════════════════════════════════════════
-- DOSSIER 2 — EFFICACITÉ (5 mesures, toujours protégées DIVIDE)
-- ════════════════════════════════════════════════════════

CTR =
DIVIDE([Total Clicks], [Total Impressions], 0)
-- ✅ Sans filtre : 2,66%
-- Format : Pourcentage 2 décimales

CPC =
DIVIDE([Total Spend], [Total Clicks], 0)
-- ✅ Sans filtre : €0,42
-- Format : Devise 2 décimales

CPM =
DIVIDE([Total Spend] * 1000, [Total Impressions], 0)
-- ✅ Sans filtre : €11,17

Cost per Conversion =
DIVIDE([Total Spend], [Total Conversions], 0)
-- ✅ Sans filtre : €10,47
-- ⚠️ Aussi appelé CPA (Cost Per Acquisition)

Conversion Rate =
DIVIDE([Total Conversions], [Total Clicks], 0)
-- ✅ Sans filtre : 4,01%

-- ════════════════════════════════════════════════════════
-- DOSSIER 3 — RENTABILITÉ (3 mesures)
-- ════════════════════════════════════════════════════════

Total Spend = SUM(performance_quotidienne[cost_eur])
-- ✅ Sans filtre : €625 488 (~€625,5K)

Conversions Value = SUM(performance_quotidienne[conversion_value_eur])
-- ✅ Sans filtre : €3 727 612 (~€3,73M)

ROAS =
DIVIDE([Conversions Value], [Total Spend], 0)
-- ✅ Sans filtre : 5,96×
-- 🎯 Seuil de rentabilité : 3× | ⚠️ < 1× = campagne à perte

### 🎓 MÉTHODE — Pourquoi 10 mesures `Prev` + 10 mesures `Delta %` ?

Une mesure de **delta** se calcule en 3 temps :

```
1. Mesure de base       → [Total Impressions] = 56,02M
2. Mesure période -1    → [Impressions Prev]   = 54,15M  (DATEADD -1 mois)
3. Delta %              → [Impressions Delta %] = (56,02 - 54,15) / 54,15 = +3,4%
```

Décomposer en 3 mesures (au lieu d'une seule mesure compacte) permet de :
- **Réutiliser `[Impressions Prev]`** dans plusieurs visuels (KPI tooltip, tableau MoM)
- **Tester chaque étape** indépendamment lors du debug
- **Bénéficier du cache VertiPaq** (la mesure Prev est calculée une seule fois)


In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 4 — PÉRIODE PRÉCÉDENTE (10 mesures)
-- Pattern : DATEADD([Date], -1, MONTH) sur la mesure courante
-- ════════════════════════════════════════════════════════

Impressions Prev =
CALCULATE([Total Impressions], DATEADD(dim_calendrier[Date], -1, MONTH))

Clicks Prev =
CALCULATE([Total Clicks], DATEADD(dim_calendrier[Date], -1, MONTH))

Conversions Prev =
CALCULATE([Total Conversions], DATEADD(dim_calendrier[Date], -1, MONTH))

CTR Prev =
CALCULATE([CTR], DATEADD(dim_calendrier[Date], -1, MONTH))

CPC Prev =
CALCULATE([CPC], DATEADD(dim_calendrier[Date], -1, MONTH))

CPM Prev =
CALCULATE([CPM], DATEADD(dim_calendrier[Date], -1, MONTH))

Cost per Conversion Prev =
CALCULATE([Cost per Conversion], DATEADD(dim_calendrier[Date], -1, MONTH))

Conversion Rate Prev =
CALCULATE([Conversion Rate], DATEADD(dim_calendrier[Date], -1, MONTH))

Total Spend Prev =
CALCULATE([Total Spend], DATEADD(dim_calendrier[Date], -1, MONTH))

Conversions Value Prev =
CALCULATE([Conversions Value], DATEADD(dim_calendrier[Date], -1, MONTH))

-- ════════════════════════════════════════════════════════
-- DOSSIER 5 — DELTAS MoM (10 mesures)
-- ════════════════════════════════════════════════════════

Impressions Delta % =
DIVIDE([Total Impressions] - [Impressions Prev], [Impressions Prev], 0)
-- Format : "+0,0%;-0,0%;0,0%" (signe forcé)

Clicks Delta % =
DIVIDE([Total Clicks] - [Clicks Prev], [Clicks Prev], 0)

Conversions Delta % =
DIVIDE([Total Conversions] - [Conversions Prev], [Conversions Prev], 0)

CTR Delta % =
DIVIDE([CTR] - [CTR Prev], [CTR Prev], 0)

CPC Delta % =
DIVIDE([CPC] - [CPC Prev], [CPC Prev], 0)
-- ⚠️ DOWN IS GOOD : baisse = bonne nouvelle

CPM Delta % =
DIVIDE([CPM] - [CPM Prev], [CPM Prev], 0)
-- ⚠️ DOWN IS GOOD

Cost per Conversion Delta % =
DIVIDE([Cost per Conversion] - [Cost per Conversion Prev], [Cost per Conversion Prev], 0)
-- ⚠️ DOWN IS GOOD

Conversion Rate Delta % =
DIVIDE([Conversion Rate] - [Conversion Rate Prev], [Conversion Rate Prev], 0)

Total Spend Delta % =
DIVIDE([Total Spend] - [Total Spend Prev], [Total Spend Prev], 0)

Conversions Value Delta % =
DIVIDE([Conversions Value] - [Conversions Value Prev], [Conversions Value Prev], 0)

In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 6 — CONVERSIONS PAR TYPE (5 mesures)
-- Filtre sur la colonne conversion_type de performance_quotidienne
-- ════════════════════════════════════════════════════════

Purchase Conversions =
CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Purchase")
-- ✅ Sans filtre : 37 727 (63% du total)

Lead Conversions =
CALCULATE(
    [Total Conversions],
    performance_quotidienne[conversion_type] IN { "Lead", "Form Submit", "Phone Call" }
)
-- ✅ Sans filtre : 16 723 (28%) — utile pour clients B2B

Signup Conversions =
CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Signup")

Demo Conversions =
CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Demo Request")

Purchase Value =
CALCULATE([Conversions Value], performance_quotidienne[conversion_type] = "Purchase")
-- ✅ Sans filtre : €2 925 935 — base du ROAS Purchase pur

-- ════════════════════════════════════════════════════════
-- DOSSIER 7 — CONTEXTUELLES (9 mesures, dont 2 nouvelles ★)
-- ════════════════════════════════════════════════════════

ROAS Verdict =
SWITCH(TRUE(),
    [ROAS] >= 5, "⭐ Tres rentable",
    [ROAS] >= 3, "✅ Rentable",
    [ROAS] >= 1, "⚠️ Limite",
    "❌ Non rentable"
)

Spend 7d MA =
AVERAGEX(
    DATESINPERIOD(dim_calendrier[Date], LASTDATE(dim_calendrier[Date]), -7, DAY),
    [Total Spend]
)

Spend YTD =
TOTALYTD([Total Spend], dim_calendrier[Date])

Spend YoY % =
VAR _curr = [Total Spend]
VAR _prev = CALCULATE([Total Spend], SAMEPERIODLASTYEAR(dim_calendrier[Date]))
RETURN DIVIDE(_curr - _prev, _prev, 0)

Nb Campagnes Actives =
CALCULATE(DISTINCTCOUNT(performance_quotidienne[campaign_id]), performance_quotidienne[clicks] > 0)

% Spend Account =
DIVIDE([Total Spend], CALCULATE([Total Spend], ALL(campaigns)), 0)

QS Moyen = AVERAGE(keywords[quality_score])
-- ✅ Sans filtre : 6,75 → arrondi à 6,8 / 10

Cost per Purchase =
DIVIDE([Total Spend], [Purchase Conversions], 0)

-- ★ NOUVELLE — Verdict ROAS en icône seule (pour tableau Page 02)
ROAS Verdict Icon =
SWITCH(TRUE(),
    [ROAS] >= 5, "⭐",
    [ROAS] >= 3, "✅",
    [ROAS] >= 1, "⚠️",
    "❌"
)

-- ★ NOUVELLE — Quartile ROAS formaté Q1/Q2/Q3/Q4 (pour tableau Page 02)
Qrt = "Q" & MAX(gads_campaigns_rank[quartile_roas])

### 🎓 MÉTHODE — Filtres complexes sur deux tables

La page **03 Keywords** demande de combiner deux conditions sur des tables différentes :
- `keywords[quality_score] <= 3` (table dimension)
- `SUM(performance_quotidienne[cost_eur]) > 100` (mesure agrégée sur table de faits)

Le pattern DAX est : `FILTER + ADDCOLUMNS + CALCULATE` pour matérialiser le spend par keyword puis filtrer.


In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 8 — KEYWORDS KPI (4 mesures, toutes nouvelles ★)
-- Utilisées par les 3 cards de la Page 03 Keywords
-- ════════════════════════════════════════════════════════

-- ★ Nb Keywords Actifs — sous-titre "287 actifs" / "328 actifs"
Nb Keywords Actifs =
CALCULATE(
    DISTINCTCOUNT(keywords[keyword_id]),
    keywords[statut] = "Enabled"
)
-- ✅ Sans filtre : 328

-- ★ Keywords A Pauser — KPI orange "23"
-- Critères de gaspillage : QS faible ET spend significatif
Keywords A Pauser =
VAR _seuil_qs = 3
VAR _seuil_spend = 100
RETURN
COUNTROWS(
    FILTER(
        ADDCOLUMNS(
            FILTER(
                keywords,
                keywords[quality_score] <= _seuil_qs
                && keywords[statut] = "Enabled"
            ),
            "_spend", CALCULATE(SUM(performance_quotidienne[cost_eur]))
        ),
        [_spend] > _seuil_spend
    )
)
-- ✅ Sans filtre : 23 (QS ≤ 3 + spend > 100€ + Enabled)

-- ★ Top CTR Keyword Valeur — KPI violet "17,0%"
Top CTR Keyword Valeur =
VAR _seuil_impressions = 1000
VAR _table_ctr =
    FILTER(
        ADDCOLUMNS(
            SUMMARIZE(
                FILTER(performance_quotidienne, NOT ISBLANK(performance_quotidienne[keyword_id])),
                performance_quotidienne[keyword_id]
            ),
            "_imp", CALCULATE(SUM(performance_quotidienne[impressions])),
            "_ctr", DIVIDE(
                CALCULATE(SUM(performance_quotidienne[clicks])),
                CALCULATE(SUM(performance_quotidienne[impressions]))
            )
        ),
        [_imp] >= _seuil_impressions
    )
RETURN MAXX(_table_ctr, [_ctr])
-- ✅ Sans filtre : 17,0%

-- ★ Top CTR Keyword Nom — sous-titre "brand hôtel sénégal p (Exact)"
Top CTR Keyword Nom =
VAR _seuil_impressions = 1000
VAR _table_ctr =
    FILTER(
        ADDCOLUMNS(
            SUMMARIZE(
                FILTER(performance_quotidienne, NOT ISBLANK(performance_quotidienne[keyword_id])),
                performance_quotidienne[keyword_id]
            ),
            "_imp", CALCULATE(SUM(performance_quotidienne[impressions])),
            "_ctr", DIVIDE(
                CALCULATE(SUM(performance_quotidienne[clicks])),
                CALCULATE(SUM(performance_quotidienne[impressions]))
            )
        ),
        [_imp] >= _seuil_impressions
    )
VAR _max_ctr = MAXX(_table_ctr, [_ctr])
VAR _top_kw_id =
    MAXX(FILTER(_table_ctr, [_ctr] = _max_ctr), performance_quotidienne[keyword_id])
VAR _kw_text = LOOKUPVALUE(keywords[keyword_text], keywords[keyword_id], _top_kw_id)
VAR _match   = LOOKUPVALUE(keywords[match_type],   keywords[keyword_id], _top_kw_id)
RETURN _kw_text & " (" & _match & ")"
-- ✅ Sans filtre : "brand hôtel sénégal p (Exact)"

-- ════════════════════════════════════════════════════════
-- DOSSIER 9 — DEVICES (3 mesures, toutes nouvelles ★)
-- Utilisées par les 3 cards de la Page 05 Breakdown
-- ════════════════════════════════════════════════════════

% Clicks Desktop =
VAR _total = CALCULATE([Total Clicks], ALL(performance_quotidienne[device]))
VAR _curr  = CALCULATE([Total Clicks], performance_quotidienne[device] = "Desktop")
RETURN DIVIDE(_curr, _total)
-- ✅ Sans filtre : 43%

% Clicks Mobile =
VAR _total = CALCULATE([Total Clicks], ALL(performance_quotidienne[device]))
VAR _curr  = CALCULATE([Total Clicks], performance_quotidienne[device] = "Mobile")
RETURN DIVIDE(_curr, _total)
-- ✅ Sans filtre : 52%

% Clicks Tablet =
VAR _total = CALCULATE([Total Clicks], ALL(performance_quotidienne[device]))
VAR _curr  = CALCULATE([Total Clicks], performance_quotidienne[device] = "Tablet")
RETURN DIVIDE(_curr, _total)
-- ✅ Sans filtre : 5%
-- Vérification : Desktop + Mobile + Tablet = 100% ✅

### 🎓 MÉTHODE — Générer du HTML en DAX

Pour la **Heatmap Jour × Heure** de la Page 05, on n'utilise pas la matrice native Power BI (limite de mise en forme conditionnelle sur cellule individuelle). On génère **une chaîne HTML complète en DAX** affichée dans un visuel **HTML Content** (AppSource).

Avantages :
- Contrôle total sur le rendu (couleurs, padding, border-radius, fond transparent)
- 56 cellules colorées en 1 seul appel → extrêmement compact
- Réutilisable pour d'autres heatmaps (clicks, CVR...)

Pattern : 2 `DATATABLE` (jours, tranches) + 2 `CONCATENATEX` imbriqués + `SWITCH` pour la palette 6 paliers.


In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 10 — HTML VISUELS (1 mesure, nouvelle ★)
-- ════════════════════════════════════════════════════════

-- ★ Heatmap Jour x Heure HTML — fond TRANSPARENT
-- 7 jours x 8 tranches de 3h, palette 6 paliers (pâle → rouge orangé)
-- À utiliser dans visuel HTML Content (AppSource)

Heatmap Jour Heure HTML =
VAR _jours =
    DATATABLE(
        "j_ord", INTEGER, "j_en", STRING, "j_fr", STRING,
        {
            {1,"Monday",   "Lun"},
            {2,"Tuesday",  "Mar"},
            {3,"Wednesday","Mer"},
            {4,"Thursday", "Jeu"},
            {5,"Friday",   "Ven"},
            {6,"Saturday", "Sam"},
            {7,"Sunday",   "Dim"}
        }
    )
VAR _tranches =
    DATATABLE(
        "ord", INTEGER, "h_min", INTEGER, "h_max", INTEGER, "label", STRING,
        {
            {1,0,3,"0-3"},   {2,3,6,"3-6"},   {3,6,9,"6-9"},    {4,9,12,"9-12"},
            {5,12,15,"12-15"},{6,15,18,"15-18"},{7,18,21,"18-21"},{8,21,24,"21-24"}
        }
    )
VAR _td_base = "text-align:center;padding:6px 8px;font-weight:500;font-size:11px;border-radius:3px;"
VAR _th_base = "text-align:center;padding:6px 8px;color:#888;font-weight:500;font-size:11px;background:transparent;border:none;"
VAR _day_base = "text-align:right;padding:6px 10px;color:#1F1F1F;font-weight:600;font-size:11px;background:transparent;border:none;"
VAR _thead =
    "<tr><th style='" & _th_base & "'></th>" &
    CONCATENATEX(_tranches, "<th style='" & _th_base & "'>" & [label] & "</th>", "", [ord], ASC) &
    "</tr>"
VAR _tbody =
    CONCATENATEX(
        _jours,
        VAR _j_en_curr = [j_en]
        VAR _j_fr_curr = [j_fr]
        VAR _cells =
            CONCATENATEX(
                _tranches,
                VAR _hmin = [h_min]
                VAR _hmax = [h_max]
                VAR _v =
                    CALCULATE(
                        [Total Spend] / 1000,
                        performance_quotidienne[day_of_week] = _j_en_curr,
                        FILTER(
                            ALL(performance_quotidienne[hour]),
                            performance_quotidienne[hour] >= _hmin && performance_quotidienne[hour] < _hmax
                        )
                    )
                VAR _bg =
                    SWITCH(TRUE(),
                        _v < 2,  "#F4F4FB",
                        _v < 6,  "#E8E5F8",
                        _v < 12, "#FCE7C5",
                        _v < 18, "#FFB573",
                        _v < 22, "#F58F4F",
                                 "#E25F2F"
                    )
                VAR _fg = IF(_v < 18, "#1F1F1F", "#FFFFFF")
                RETURN
                    "<td style='" & _td_base & "background:" & _bg & ";color:" & _fg & "'>" &
                    FORMAT(_v, "0.0") & "</td>",
                "", [ord], ASC
            )
        RETURN
            "<tr><td style='" & _day_base & "'>" & _j_fr_curr & "</td>" & _cells & "</tr>",
        "", [j_ord], ASC
    )
RETURN
"<div style='font-family:Segoe UI,Arial,sans-serif;padding:8px;background:transparent;'>" &
"<table style='border-collapse:separate;border-spacing:3px;width:100%;'>" &
"<thead>" & _thead & "</thead>" &
"<tbody>" & _tbody & "</tbody>" &
"</table>" &
"</div>"
-- ✅ Sortie : ~10 380 caractères de HTML, fond transparent

### 🎓 MÉTHODE — Helpers de couleur pour KPI cards avec deltas

Pour reproduire l'effet **"flèche verte/rouge selon performance"** des KPI cards de la Page 01, on crée **10 mesures helpers** qui retournent un code hexadécimal (`#1D9E75` vert ou `#E25F2F` rouge) selon le signe du delta.

**Logique métier appliquée :**

| Type | Mesures concernées | Règle | Si delta ≤ 0 | Si delta > 0 |
|---|---|---|---|---|
| **UP-IS-GOOD** ✅ | Impressions, Clicks, Conversions, CTR, Conversion Rate, Conversions Value | hausse = bon | 🔴 Rouge `#E25F2F` | 🟢 Vert `#1D9E75` |
| **DOWN-IS-GOOD** 🔻 | CPC, Total Spend, CPM, Cost per Conversion | baisse de coût = bon | 🟢 Vert `#1D9E75` | 🔴 Rouge `#E25F2F` |

Ces 10 mesures sont stockées dans le dossier **`_Helpers`** et utilisées dans les KPI cards via la **mise en forme conditionnelle "Format selon : Valeur de champ"** (cf. Page 01 Overview).


In [ ]:
-- ════════════════════════════════════════════════════════
-- DOSSIER 11 — _HELPERS (10 mesures de couleur conditionnelle)
-- À utiliser dans la mise en forme conditionnelle des KPI cards
-- Format → Étiquettes de données → Couleur → fx → Format selon : Valeur de champ
-- ════════════════════════════════════════════════════════

-- ─── UP-IS-GOOD : hausse = vert, baisse = rouge (6 mesures) ───

Color Impressions Delta =
IF([Impressions Delta %] >= 0, "#1D9E75", "#E25F2F")

Color Clicks Delta =
IF([Clicks Delta %] >= 0, "#1D9E75", "#E25F2F")

Color Conversions Delta =
IF([Conversions Delta %] >= 0, "#1D9E75", "#E25F2F")

Color CTR Delta =
IF([CTR Delta %] >= 0, "#1D9E75", "#E25F2F")

Color Conversion Rate Delta =
IF([Conversion Rate Delta %] >= 0, "#1D9E75", "#E25F2F")

Color Conversions Value Delta =
IF([Conversions Value Delta %] >= 0, "#1D9E75", "#E25F2F")

-- ─── DOWN-IS-GOOD : baisse = vert (4 mesures de coût) ───

Color CPC Delta =
IF([CPC Delta %] <= 0, "#1D9E75", "#E25F2F")

Color Spend Delta =
IF([Total Spend Delta %] <= 0, "#1D9E75", "#E25F2F")

Color CPM Delta =
IF([CPM Delta %] <= 0, "#1D9E75", "#E25F2F")

Color Cost per Conversion Delta =
IF([Cost per Conversion Delta %] <= 0, "#1D9E75", "#E25F2F")

-- ✅ Validation : sur la période sans filtre, [Total Spend Delta %] = +6,9%
--    → [Color Spend Delta] retourne "#E25F2F" (rouge) car on a dépensé plus.

### 💡 INTERPRÉTATION — Tableau de validation des KPIs principaux

Avant de construire les pages, **valide les 12 KPIs principaux** dans une page brouillon en glissant une carte par mesure :

| Mesure | Valeur attendue | Page d'usage |
|---|---|---|
| `[Total Impressions]` | **56,02M** | 01 Overview |
| `[Total Clicks]` | **1,49M** | 01 Overview |
| `[Total Conversions]` | **59,7K** | 01 Overview, 04 Conversions |
| `[CTR]` | **2,66%** | 01 Overview |
| `[CPC]` | **€0,42** | 01 Overview |
| `[Total Spend]` | **€625,5K** | 01 Overview, 05 Breakdown |
| `[Conversions Value]` | **€3,73M** | 01 Overview |
| `[ROAS]` | **5,96×** | 02 Campaigns |
| `[QS Moyen]` | **6,7 / 10** | 03 Keywords |
| `[Keywords A Pauser]` | **23** | 03 Keywords |
| `[Top CTR Keyword Valeur]` | **17,0%** | 03 Keywords |
| `[% Clicks Mobile]` | **52%** | 05 Breakdown |

### 🏢 MÉTIER — Lecture rapide de ces KPIs

> **CTR à 2,66%** = dans la fourchette Google Ads Search (2-4%). Pas d'alerte.
> **ROAS à 5,96×** = très bon, la moitié des comptes à plus de 5× (seuil "Très rentable").
> **CPC à €0,42** = très compétitif pour des campagnes Brand (où le CPC tourne autour de €0,30-€0,50 sur le marché Afrique de l'Ouest).
> **Cost/Conv à €10,47** = correct vu le mix produit (e-commerce + B2B).


---
## Étape 4 — Construction des 5 pages

### 📊 Page 01 — Overview

**Objectif :** répondre à *« comment se comporte mon Paid Media ce mois-ci ? »* en **30 secondes**.

#### Layout

```
┌─────────────────────────────────────────────────────────────────────────┐
│  G  GoogleAdsPulse · Paid Media Analytics                               │
│ logo │ Overview · Campaigns · Keywords · Conversions · Breakdown │📅📅 │  ← header
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Overview                                                               │
│  Vue globale des 5 comptes · Performance 24 mois                        │
│                                                                         │
│  ┌────────────┬────────────┬────────────┬────────────┬────────────┐   │
│  │👁 Impress. │🎯 Clicks   │🎯 Convers. │📈 CTR      │🪙 CPC      │   │ ← Ligne 1 :
│  │ 56,02M     │ 1,49M      │ 59,7K      │ 2,66%      │ 0,42 €     │   │   Volume +
│  │ +3,4% ↑🟢  │ -11,4% ↓🔴 │ -39,0% ↓🔴 │ -14,3% ↓🔴 │ -0,9% ↑🟢  │   │   efficacité
│  └────────────┴────────────┴────────────┴────────────┴────────────┘   │
│  ┌────────────┬────────────┬────────────┬────────────┬────────────┐   │
│  │🪙 Spend    │🪙 CPM      │⚠️ Cost/Conv│📈 Conv rate│📈 Conv val │   │ ← Ligne 2 :
│  │ 625,8K€    │ 11,17 €    │ 10,48 €    │ 4,01%      │ 3,73M€     │   │   Coûts +
│  │ -12,3% ↑🟢 │ -15,2% ↑🟢 │ +43,9% ↓🔴 │ -31,2% ↓🔴 │ -59,9% ↓🔴 │   │   rentabilité
│  └────────────┴────────────┴────────────┴────────────┴────────────┘   │
│                                                                         │
│  ┌──────────────────────┬──────────────────────┐                       │
│  │ Impressions vs Clicks│ CPC evolution (€)    │                       │
│  │ (combo bar+line,     │ (line chart orange)  │                       │
│  │  axe Y secondaire)   │                      │                       │
│  └──────────────────────┴──────────────────────┘                       │
│  ┌──────────────────────┬──────────────────────┐                       │
│  │ Spend amount by Date │ Conversions          │                       │
│  │ (area chart pêche)   │ mensuelles (K)       │                       │
│  │                      │ (line chart vert)    │                       │
│  └──────────────────────┴──────────────────────┘                       │
└─────────────────────────────────────────────────────────────────────────┘
```

### 🎯 Construction des 10 KPI cards avec deltas (le cœur de la page)

#### 🎓 MÉTHODE — Approche "Carte native + helper de couleur"

Power BI ne sait pas afficher 2 textes superposés dans **une seule carte native**. Pour reproduire le mockup avec **valeur principale + delta coloré + flèche**, on combine :

1. **Un conteneur Rectangle** (fond blanc, coins arrondis 8px, ombre légère)
2. **Une icône** (image PNG Lucide, ~32px à gauche)
3. **Une zone de texte** pour le label gris ("Impressions", "CPC", ...)
4. **Visuel Carte 1** pour la valeur principale (`[Total Impressions]`, etc.)
5. **Visuel Carte 2** pour le delta % avec **format string magique** + couleur conditionnelle

```
┌─────────────────────────────────┐
│  👁  Impressions                │  ← icône (image) + label (zone texte)
│  56,02M                         │  ← Visuel Carte 1 (valeur principale)
│  +3,4% ↑                        │  ← Visuel Carte 2 (delta + couleur conditionnelle)
└─────────────────────────────────┘
```

#### 🪄 L'astuce du format string `"+0,0% ↑;-0,0% ↓;0,0%"`

Power BI accepte 3 sections séparées par `;` dans un format string : **positif ; négatif ; zéro**. On peut y intégrer **n'importe quel caractère Unicode** (↑ ↓ ▲ ▼). La flèche est statique dans le format mais sa **couleur varie** via la mesure `[Color ... Delta]`. Pas besoin de mesure DAX dédiée pour la flèche.

#### ⚠️ Logique UP-IS-GOOD vs DOWN-IS-GOOD

Sur le mockup, **CPC -0,9% affiche ↑ vert** (la baisse de coût est une bonne nouvelle, donc flèche vers le haut). C'est de la sémantique business, pas mathématique. Pour les **4 mesures DOWN-IS-GOOD** (CPC, CPM, Spend, Cost/Conv), il faut **inverser le format string** :

| Type | Mesures | Format string delta |
|---|---|---|
| **UP-IS-GOOD** ✅ | Impressions, Clicks, Conversions, CTR, Conv. Rate, Conv. Value | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| **DOWN-IS-GOOD** 🔻 | CPC, Spend, CPM, Cost per Conversion | `"+0,0% ↓;-0,0% ↑;0,0%"` (inversé) |

> **Pourquoi cette inversion ?** Pour un coût (CPC, Spend, CPM, Cost/Conv), une **baisse** est une bonne nouvelle. La flèche ↑ (verte) doit donc apparaître quand le delta est négatif. L'inversion se fait **dans le format string** (et pas dans la mesure de couleur, qui suit aussi cette même logique inversée).

#### Tableau récapitulatif des 10 KPI cards

| # | KPI | Mesure valeur | Format valeur | Mesure delta | Helper couleur | Format string delta |
|---|---|---|---|---|---|---|
| 1 | Impressions | `[Total Impressions]` | `0.00,,"M"` → 56,02M | `[Impressions Delta %]` | `[Color Impressions Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| 2 | Clicks | `[Total Clicks]` | `0.00,,"M"` → 1,49M | `[Clicks Delta %]` | `[Color Clicks Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| 3 | Conversions | `[Total Conversions]` | `0.0,"K"` → 59,7K | `[Conversions Delta %]` | `[Color Conversions Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| 4 | CTR | `[CTR]` | `0.00%` → 2,66% | `[CTR Delta %]` | `[Color CTR Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| 5 | **CPC** ⚠️ | `[CPC]` | `0.00 €` → 0,42 € | `[CPC Delta %]` | `[Color CPC Delta]` | **`"+0,0% ↓;-0,0% ↑;0,0%"`** |
| 6 | **Spend amount** ⚠️ | `[Total Spend]` | `0,"K€"` → 625,8K€ | `[Total Spend Delta %]` | `[Color Spend Delta]` | **`"+0,0% ↓;-0,0% ↑;0,0%"`** |
| 7 | **CPM** ⚠️ | `[CPM]` | `0.00 €` → 11,17 € | `[CPM Delta %]` | `[Color CPM Delta]` | **`"+0,0% ↓;-0,0% ↑;0,0%"`** |
| 8 | **Cost / Conv** ⚠️ | `[Cost per Conversion]` | `0.00 €` → 10,48 € | `[Cost per Conversion Delta %]` | `[Color Cost per Conversion Delta]` | **`"+0,0% ↓;-0,0% ↑;0,0%"`** |
| 9 | Conversion rate | `[Conversion Rate]` | `0.00%` → 4,01% | `[Conversion Rate Delta %]` | `[Color Conversion Rate Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |
| 10 | Conv. value | `[Conversions Value]` | `0.00,,"M€"` → 3,73M€ | `[Conversions Value Delta %]` | `[Color Conversions Value Delta]` | `"+0,0% ↑;-0,0% ↓;0,0%"` |

⚠️ = mesures DOWN-IS-GOOD, format string inversé.

#### Spec d'icône par KPI

| KPI | Icône Lucide | Couleur icône |
|---|---|---|
| 1 — Impressions | `eye` | Violet `#7B61FF` |
| 2 — Clicks | `mouse-pointer-2` | Violet |
| 3 — Conversions | `target` | Violet |
| 4 — CTR | `trending-up` | Violet |
| 5 — CPC | `coins` | Violet |
| 6 — Spend amount | `coins` | Violet |
| 7 — CPM | `coins` | Violet |
| 8 — Cost / Conv | `alert-triangle` | **Orange `#EF9F27`** |
| 9 — Conversion rate | `trending-up` | Violet |
| 10 — Conv. value | `trending-up` | Violet |

#### 🛠 Procédure pas-à-pas pour 1 KPI card (à répliquer 10 fois)

```
1. Insertion → Forme Rectangle
   └── Fond blanc, coins arrondis 8px, ombre légère
   └── Taille : ~280×95 px

2. Insertion → Image (icône Lucide PNG, ~32px à gauche)

3. Insertion → Zone de texte (label gris 12pt, ex. "Impressions")

4. Insertion → Visuel Carte (valeur principale)
   ├── Champ : [Total Impressions]
   ├── Format string : 0.00,,"M"
   ├── Couleur : noir #1F1F1F, gras 22pt
   └── Étiquette de catégorie : DÉSACTIVÉE

5. Insertion → Visuel Carte (delta, juste en dessous)
   ├── Champ : [Impressions Delta %]
   ├── Format string PERSONNALISÉ : "+0,0% ↑;-0,0% ↓;0,0%"
   ├── Police : 11pt, gras
   ├── Étiquette de catégorie : DÉSACTIVÉE
   └── Couleur conditionnelle :
       Format → Élément visuel → Légende → Étiquettes de données → Couleur → fx
       └── Format selon : Valeur de champ
           └── Champ basé sur : [Color Impressions Delta]

6. Sélectionne les 5 éléments → Ctrl+G pour grouper

7. Ctrl+C / Ctrl+V pour dupliquer 9 fois → tu obtiens 10 cards identiques

8. Sur chaque copie, change uniquement :
   - L'icône (image)
   - Le label (zone de texte)
   - Les champs des 2 cartes (valeur + delta)
   - Le format string delta (DOWN-IS-GOOD vs UP-IS-GOOD)
   - La couleur conditionnelle (sélectionner le bon [Color ... Delta])
```

### 📈 Visuel principal — Impressions vs Clicks (combo dual-axis)

> **🎓 MÉTHODE — Pourquoi un visuel à 2 axes Y ?**
>
> Les Impressions sont **~37× plus grandes** que les Clicks (4,7M vs 125K en moyenne). Avec une seule échelle Y, la courbe des Clicks est complètement écrasée → ligne plate visuelle. La solution : **"Histogramme groupé et graphique en courbes"** avec axe Y secondaire.

```
Visuel : Histogramme groupé et graphique en courbes
├── Axe X
│   └── dim_calendrier[Mois_Nom] (trier par Mois_Num)
├── Axe Y des colonnes
│   └── [Total Impressions]    ← grandes valeurs en colonnes bleu clair #5DADE2
└── Axe Y des lignes (secondaire)
    └── [Total Clicks]         ← petites valeurs en ligne navy #1F3864
```

**Format → Axe Y → Afficher l'axe secondaire → Activé.** Format primaire `0.0,,"M"` (4M, 5M), format secondaire `0,"K"` (125K, 191K).

### Les 3 autres visuels

| Visuel | Type | Champs | Format | Couleur |
|---|---|---|---|---|
| **CPC evolution** | Line chart | X: Mois_Nom · Y: `[CPC]` | `€0.00` | Orange `#EF9F27`, marqueurs ronds, étiquettes activées |
| **Spend amount by Date** | Aire (Area) | X: Date · Y: `[Total Spend] / 1000` | `0,"K€"` | Pêche `#FFD4B5` rempli, aucune étiquette |
| **Conversions mensuelles** | Line chart | X: Mois_Nom · Y: `[Total Conversions] / 1000` | `0.0,"K"` | Vert `#1D9E75`, marqueurs ronds, étiquettes activées |


In [ ]:
-- Requête DAX de validation Page 01 — KPIs et timelines

EVALUATE
ROW(
    "Impressions",  [Total Impressions],
    "Clicks",       [Total Clicks],
    "Conversions",  [Total Conversions],
    "CTR",          [CTR],
    "CPC",          [CPC],
    "Spend",        [Total Spend],
    "CPM",          [CPM],
    "CostConv",     [Cost per Conversion],
    "ConvRate",     [Conversion Rate],
    "ConvValue",    [Conversions Value]
)
-- Attendu : 56 018 776 | 1 488 092 | 59 715 | 0,0266 | 0,42 |
--           625 488 | 11,17 | 10,47 | 0,0401 | 3 727 612

-- Évolution mensuelle Impressions vs Clicks 2024
EVALUATE
SUMMARIZECOLUMNS(
    dim_calendrier[Mois_Num],
    dim_calendrier[Mois_Nom],
    "Impressions", [Total Impressions],
    "Clicks",      [Total Clicks],
    "Ratio",       DIVIDE([Total Impressions], [Total Clicks])
)
ORDER BY dim_calendrier[Mois_Num]
-- Vérif : ratio Imp/Clk constant ~37× (pic Q4 +56% en décembre)

### 🎯 Page 02 — Campaigns

**Objectif :** identifier les **campagnes qui rapportent** vs celles qui drainent le budget.

#### Layout

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Campaigns                                                              │
│  25 campagnes actives · Performance par ROAS                            │
│                                                                         │
│  ┌──────────────────────────┬──────────────────────────┐               │
│  │ ⭐ Top 5 campaigns by    │ ⭐ Bottom 5 campaigns by │               │
│  │    ROAS (vert)           │    ROAS (rouge)          │               │
│  │ Brand-MedSupply  19,19×  │ Search-MedicalEquip 0,30×│               │
│  │ Brand-Hotels-CI  18,96×  │ Shopping-Packages   0,37×│               │
│  │ Brand-Search-CI  12,31×  │ PerfMax-Inscriptions0,68×│               │
│  │ Brand-Search-Edt 11,59×  │ Display-Remarketing 1,94×│               │
│  │ Brand-Search-Bnk 11,40×  │ Search-Generic-Tech 2,60×│               │
│  └──────────────────────────┴──────────────────────────┘               │
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │ 📊 Toutes les campagnes — Classement et verdict                 │   │
│  │ Campaign | Type | Spend | Clicks | Conv | CTR | Cost/Conv |     │   │
│  │ ROAS | ROAS Verdict | Qrt                                        │   │
│  │ ───── 25 lignes triables ─────                                   │   │
│  └─────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────┘
```

#### 🏆 Top 5 / Bottom 5 ROAS — Specs

| Élément | Top 5 | Bottom 5 |
|---|---|---|
| Type visuel | Bar chart horizontal | Bar chart horizontal |
| Y | `campaigns[campaign_name]` | `campaigns[campaign_name]` |
| X | `[ROAS]` | `[ROAS]` |
| Tri | Décroissant TOP 5 | Croissant BOTTOM 5 |
| Couleur | Vert `#1D9E75` | Rouge `#E25F2F` |
| Format | `0.00"×"` → 19,19× | idem |
| Filtre TopN | Top 5 par ROAS | Bottom 5 par ROAS |
| Étiquettes données | À droite des barres, gras | Idem |

> **🎓 MÉTHODE — Filtre TopN dans Power BI**
>
> Sur le visuel → panneau **Filtres** (à droite) → glisser `[ROAS]` dans "Filtres de ce visuel" → Type de filtre : **Top N** → 5 → Par valeur : `[ROAS]` (ou Bottom N pour le bas).

#### 📊 Tableau classement complet — Specs

```
Visuel : Tableau natif Power BI
├── Campaign       → campaigns[campaign_name]              ⚠️ NE PAS utiliser gads_campaigns_rank[campaign_name]
├── Type           → campaigns[campaign_type]
├── Spend          → [Total Spend]                         format €0,"K€"
├── Clicks         → [Total Clicks]                        format 0,"K"
├── Conversions    → [Total Conversions]                   format 0.0,"K"
├── CTR            → [CTR]                                  format 0.0%
├── Cost/Conv      → [Cost per Conversion]                 format €0.00
├── ROAS           → [ROAS]                                  format 0.00"×"
├── ROAS Verdict   → [ROAS Verdict Icon]   ★ NOUVELLE
└── Qrt            → [Qrt]                  ★ NOUVELLE
```

> **⚠️ POINT CRITIQUE — Quelle table mettre en première colonne ?**
>
> Tu **dois utiliser `campaigns[campaign_name]`** (la vraie dimension) en première colonne. Si tu mets `gads_campaigns_rank[campaign_name]`, les mesures Spend/Clicks/Conv/ROAS ne se filtrent pas par campagne et tu obtiens **les mêmes valeurs sur toutes les lignes** (bug classique).
>
> La mesure `[Qrt]` fonctionne car elle utilise `MAX(gads_campaigns_rank[quartile_roas])` — Power BI fait la jointure implicite via `campaign_id` (relation existante entre les 2 tables).

#### Mise en forme du tableau

| Élément | Réglage |
|---|---|
| En-tête de ligne | Fond gris pâle `#F5F5F5`, gras navy |
| Lignes alternées | Activées (zébrures gris très clair) |
| Tri par défaut | ROAS décroissant |
| Colonne Verdict | Largeur réduite (~80px), centrée |
| Colonne Qrt | Largeur réduite (~50px), centrée |
| Totaux | Désactivés (chaque ligne est une moyenne pondérée) |


In [ ]:
-- Requête DAX de validation Page 02

-- 1. Top 5 et Bottom 5 ROAS
EVALUATE
TOPN(
    5,
    SUMMARIZECOLUMNS(
        campaigns[campaign_name],
        "ROAS", [ROAS]
    ),
    [ROAS], DESC
)
-- Attendu : Brand-MedSupply 19,19 / Brand-Hotels-CI 18,96 / Brand-Search-CI 12,31 /
--           Brand-Search-EdTech 11,59 / Brand-Search-Bank 11,40

-- 2. Tableau complet avec Verdict + Qrt
EVALUATE
SUMMARIZECOLUMNS(
    campaigns[campaign_name],
    campaigns[campaign_type],
    "Spend",     [Total Spend],
    "Clicks",    [Total Clicks],
    "Conv",      [Total Conversions],
    "CTR",       [CTR],
    "CostConv",  [Cost per Conversion],
    "ROAS",      [ROAS],
    "Verdict",   [ROAS Verdict Icon],
    "Qrt",       [Qrt]
)
ORDER BY [ROAS] DESC

### 🏢 MÉTIER — Lecture des Top/Bottom ROAS

> ⭐ **Top 5 = toutes des campagnes Brand** (Brand-MedSupply 19,19× · Brand-Hotels-CI 18,96×). C'est attendu : les keywords marque ont un Quality Score élevé (~9-10), un CPC très bas et un taux de conversion fort. Action : **augmenter les budgets Brand de 30%** sans changer la stratégie d'enchères.
>
> ❌ **Bottom 5 — diagnostic à faire pour chacune** :
> - **Search-MedicalEquip 0,30×** → CPC trop élevé sur des keywords génériques B2B. À pauser ou restructurer en match exact strict.
> - **Shopping-Packages 0,37×** → marges trop fines sur les forfaits voyage low-cost. Repositionner sur des packages premium.
> - **PerfMax-Inscriptions 0,68×** → conversions Signup mal valorisées. Ajuster la valeur de conversion en upload offline.


### 🔑 Page 03 — Keywords

**Objectif :** détecter les **keywords champions** à valoriser et les **gaspillages** à pauser.

#### Layout

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Keywords                                                               │
│  372 mots-clés · Performance et rétention dans le temps                 │
│                                                                         │
│  ┌──────────────────┬──────────────────┬──────────────────┐            │
│  │ ✅ Quality Score │ ⏸ Keywords à    │ ⚡ Top CTR       │            │
│  │   moyen           │   pauser          │   keyword         │            │
│  │   6,7             │   23              │   17,0%           │            │
│  │   328 actifs      │ QS≤3+spend>100€   │ brand hôtel...    │            │
│  └──────────────────┴──────────────────┴──────────────────┘            │
│                                                                         │
│  🔥 Cohort Retention — Keywords par mois du 1er clic                   │
│  Lignes = cohorte d'apparition · Colonnes = mois relatif                │
│                                                                         │
│       Month  M0   M1   M2  ...  M12                                    │
│      2023-01 100  100  100 ...   91                                    │
│      2023-08 100  100  100 ...  100                                    │
│      2023-09 100  100  100 ...  100                                    │
│      2023-10 100  100  100 ...  100                                    │
└─────────────────────────────────────────────────────────────────────────┘
```

#### 🎯 Specs des 3 cards (approche composée)

Chaque card = **conteneur blanc** + **icône colorée** (~64px à gauche) + **label gris** + **valeur grosse colorée** + **sous-titre dynamique**.

| Card | Icône (Lucide) | Couleur | Label | Valeur | Sous-titre |
|---|---|---|---|---|---|
| 🟢 **QS moyen** | `check-circle` | Vert `#1D9E75` | "Quality Score moyen" | `[QS Moyen]` format `0.0` → 6,7 | `[Nb Keywords Actifs]` format `0" actifs"` → 328 actifs |
| 🟠 **À pauser** | `pause` | Orange `#EF9F27` | "Keywords à pauser" | `[Keywords A Pauser]` → 23 | "QS ≤ 3 + spend > 100€" *(statique)* |
| 🟣 **Top CTR** | `zap` | Violet `#7B61FF` | "Top CTR keyword" | `[Top CTR Keyword Valeur]` format `0.0%` → 17,0% | `[Top CTR Keyword Nom]` → brand hôtel sénégal p (Exact) |

#### 🔥 Cohort Retention — Construction

Cette matrice nécessite une **table calculée DAX** (Modélisation → Nouvelle table) qui transforme les données brutes en format long [cohorte, mois_relatif, retention_pct] :

```dax
gads_cohort_long =
VAR _kw_cohort =
    ADDCOLUMNS(
        VALUES(performance_quotidienne[keyword_id]),
        "cohort_date",
            VAR _first = CALCULATE(
                MIN(performance_quotidienne[date]),
                performance_quotidienne[clicks] > 0
            )
            RETURN DATE(YEAR(_first), MONTH(_first), 1)
    )
VAR _activite =
    DISTINCT(
        SELECTCOLUMNS(
            FILTER(performance_quotidienne, performance_quotidienne[clicks] > 0),
            "kw_id_a", performance_quotidienne[keyword_id],
            "mois_actif", DATE(YEAR(performance_quotidienne[date]), MONTH(performance_quotidienne[date]), 1)
        )
    )
VAR _joined =
    NATURALINNERJOIN(
        SELECTCOLUMNS(_kw_cohort, "kw_id_a", performance_quotidienne[keyword_id], "cohort_date", [cohort_date]),
        _activite
    )
VAR _with_relatif =
    SELECTCOLUMNS(
        _joined,
        "kw_id", [kw_id_a],
        "cohort_date", [cohort_date],
        "mois_relatif", DATEDIFF([cohort_date], [mois_actif], MONTH)
    )
VAR _agg_actifs =
    GROUPBY(
        FILTER(_with_relatif, [mois_relatif] <= 12),
        [cohort_date], [mois_relatif],
        "nb_actifs", COUNTX(CURRENTGROUP(), [kw_id])
    )
VAR _agg_taille =
    GROUPBY(
        _kw_cohort,
        [cohort_date],
        "taille", COUNTX(CURRENTGROUP(), performance_quotidienne[keyword_id])
    )
RETURN
ADDCOLUMNS(
    _agg_actifs,
    "taille_cohorte",
        VAR _curr = [cohort_date]
        RETURN MAXX(FILTER(_agg_taille, [cohort_date] = _curr), [taille]),
    "retention_pct",
        VAR _curr = [cohort_date]
        VAR _t = MAXX(FILTER(_agg_taille, [cohort_date] = _curr), [taille])
        RETURN ROUND(DIVIDE([nb_actifs], _t) * 100, 0),
    "cohort_label", FORMAT([cohort_date], "yyyy-MM"),
    "mois_relatif_label", "M" & [mois_relatif]
)
```

#### 📊 Setup de la matrice Cohort Retention

```
Visuel : Matrice
├── Lignes      → gads_cohort_long[cohort_label]      (2023-01, 2023-08...)
├── Colonnes    → gads_cohort_long[mois_relatif_label] (M0, M1, ..., M12)
└── Valeurs     → SUM(gads_cohort_long[retention_pct])
                  Format : 0   |   Agrégation = Maximum (1 valeur par cellule)
```

> **🎓 ASTUCE — Tri des colonnes M0 → M12**
>
> Pour que `M0, M1, ..., M12` apparaissent dans le bon ordre (et pas alphabétique : M0, M1, M10, M11, M12, M2...), trie `mois_relatif_label` par la colonne numérique : Vue Données → sélectionne `mois_relatif_label` → Outils de colonnes → **Trier par colonne** → `mois_relatif`.

#### 🎨 Mise en forme conditionnelle (5 paliers de couleur)

Format → Cellules → Couleurs d'arrière-plan → Conditionnelle → Par règles :

| Règle | Si valeur | Couleur fond | Couleur texte |
|---|---|---|---|
| 1 | `>=` 80 et `<=` 100 | Vert foncé `#1D9E75` | Blanc |
| 2 | `>=` 65 et `<` 80 | Vert clair `#A8D8A0` | Noir |
| 3 | `>=` 55 et `<` 65 | Jaune `#FBE89A` | Noir |
| 4 | `>=` 45 et `<` 55 | Orange `#F4A261` | Noir |
| 5 | `>=` 0 et `<` 45 | Rouge clair `#E76F6F` | Blanc |

### 💡 INTERPRÉTATION — Cohort Retention sur tes données

| Cohorte | Taille | M0 | M6 | M12 |
|---|---|---|---|---|
| **2023-01** | **127 kw** | 100 | **98** | **91** |
| 2023-08 | 13 kw | 100 | 100 | 100 |
| 2023-09 | 9 kw | 100 | 100 | 100 |
| 2023-10 | 11 kw | 100 | 100 | 100 |

> 🟢 **Très bonne rétention globale** : la cohorte janvier 2023 (127 keywords, 76% du portefeuille) garde **91% de rétention à 12 mois**. Les keywords brand sont quasi-permanents.
>
> 📉 **Léger décrochage à M6-M7** : on passe de 100% à 92% (perte de 10 keywords) entre le 6e et le 7e mois. À investiguer côté Marc-Aurèle (changement d'enchères ? optimisation Quality Score ?).


In [ ]:
-- Requête DAX de validation Page 03 — Keywords KPI

EVALUATE
ROW(
    "QS Moyen",         [QS Moyen],
    "Nb Keywords Actifs", [Nb Keywords Actifs],
    "Keywords A Pauser",[Keywords A Pauser],
    "Top CTR Valeur",   [Top CTR Keyword Valeur],
    "Top CTR Nom",      [Top CTR Keyword Nom]
)
-- Attendu : 6,75 | 328 | 23 | 17,0% | brand hôtel sénégal p (Exact)

-- Vérif Cohort Retention
EVALUATE
gads_cohort_long
ORDER BY [cohort_date] ASC, [mois_relatif] ASC
-- Attendu : 4 cohortes (2023-01, 2023-08, 2023-09, 2023-10) x 13 mois (M0-M12)

### 🎯 Page 04 — Conversions

**Objectif :** comprendre **d'où viennent les conversions** et **combien elles coûtent** par type.

#### Layout

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Conversions                                                            │
│  59 712 conversions · Répartition par type et coût unitaire             │
│                                                                         │
│  ┌──────────────────────────┬──────────────────────────┐               │
│  │ Conversions par type     │ Cost per Conversion      │               │
│  │ (donut)                  │ par type (€) — bar       │               │
│  │ Purchase 63%, Lead 17%   │ Signup €13,26 / Demo...  │               │
│  │ Signup 7%, Form 7%...    │ Lead €7,53 (cheapest)    │               │
│  └──────────────────────────┴──────────────────────────┘               │
│  ┌──────────────────────────┬──────────────────────────┐               │
│  │ Conversion Value         │ Top 8 campagnes par      │               │
│  │ par type (K€) — bar vert │ nombre de conversions    │               │
│  │ Purchase €2 925,93K      │ (tableau)                │               │
│  │ Lead €468,63K, ...       │ PerfMax-LeadGen 9 110... │               │
│  └──────────────────────────┴──────────────────────────┘               │
└─────────────────────────────────────────────────────────────────────────┘
```

#### Specs des 4 visuels

| # | Visuel | Type | Champs |
|---|---|---|---|
| 1 | **Conversions par type** | Donut | Légende: `performance_quotidienne[conversion_type]` · Valeurs: `[Total Conversions]` · Centre vide 70% · Couleurs : Purchase violet, Lead vert, Signup bleu, Form Submit jaune, Phone Call rouge, Demo rose pâle |
| 2 | **Cost per Conversion par type** | Bar horizontal | Y: conversion_type · X: `[Cost per Conversion]` · Tri décroissant · Couleur uniforme violet `#7B61FF` · Format `€0.00` |
| 3 | **Conversion Value par type** | Bar vertical | X: conversion_type · Y: `[Conversions Value] / 1000` · Tri décroissant · Couleur uniforme vert `#1D9E75` · Format `€0,"K"` · Étiquettes activées |
| 4 | **Top 8 campagnes par conversions** | Tableau | `campaigns[campaign_name]` · `campaigns[campaign_type]` · `[Total Conversions]` · `[Conversions Value]` · TopN 8 par Conversions DESC |

> **🎓 MÉTHODE — Pourquoi un donut plutôt qu'un camembert ?**
>
> Le donut (anneau) laisse un espace central qu'on peut utiliser pour afficher le **total** ou un message clé. Plus moderne et plus lisible. Le pourcentage est imprimé à l'intérieur des arcs (Format → Étiquettes détaillées → "Pourcentage du total").

### 💡 INTERPRÉTATION — Lecture du mix de conversions

> **63% Purchase** confirme que le portefeuille agence est dominé par l'**e-commerce** (TechShop, AfriHotels). Les campagnes Brand drainent les conversions rapides.
>
> **Lead = CPA le moins cher (€7,53)** vs Signup à €13,26 → presque 2× plus cher. C'est cohérent : un Lead (formulaire) demande moins d'engagement utilisateur qu'un Signup (création de compte). Recommandation : pour les nouveaux clients B2B, **commencer par optimiser sur Lead** avant de passer sur Signup.

### 🏢 MÉTIER — La règle des 3 actions

À partir de cette page, Marc-Aurèle peut prendre 3 décisions concrètes :

1. **Augmenter de 25% le budget Purchase** sur les Top 4 campagnes du tableau (PerfMax-LeadGen, PerfMax-Bookings, PerfMax-AllProducts, Shopping-Electronics) — elles cumulent 26 154 conversions / 59 712 = 44% du total.
2. **Réduire les enchères Signup** de 30% car CPA trop élevé (€13,26 vs €7,53 pour Lead) — gain estimé €4 200/mois.
3. **Inviter à upload des conversions hors-ligne** pour les Demo Request (3% du volume) afin de mieux les valoriser et améliorer la machine learning Google.


In [ ]:
-- Requête DAX de validation Page 04

-- 1. Donut conversions par type
EVALUATE
SUMMARIZECOLUMNS(
    performance_quotidienne[conversion_type],
    "Conversions", [Total Conversions],
    "Pct", DIVIDE([Total Conversions], CALCULATE([Total Conversions], ALL(performance_quotidienne[conversion_type])))
)
ORDER BY [Conversions] DESC
-- Attendu : Purchase 63% | Lead 17% | Signup 7% | Form Submit 7% | Phone Call 4% | Demo 3%

-- 2. Cost per Conversion par type
EVALUATE
SUMMARIZECOLUMNS(
    performance_quotidienne[conversion_type],
    "Cost per Conv", [Cost per Conversion]
)
ORDER BY [Cost per Conv] DESC
-- Attendu : Signup €13,26 | Demo €13,17 | Purchase €11,27 | Phone €7,84 | Form €7,80 | Lead €7,53

-- 3. Top 8 campagnes par conversions
EVALUATE
TOPN(
    8,
    SUMMARIZECOLUMNS(
        campaigns[campaign_name],
        campaigns[campaign_type],
        "Conversions", [Total Conversions],
        "Value",       [Conversions Value]
    ),
    [Total Conversions], DESC
)
-- Attendu top 3 : PerfMax-LeadGen 9 110 / PerfMax-Bookings 6 733 / PerfMax-AllProducts 6 150

### 📊 Page 05 — Breakdown

**Objectif :** segmenter le spend par **device, horaire, géographie, type de campagne** pour identifier les leviers d'optimisation.

#### Layout

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Breakdown                                                              │
│  Analyse par device, horaire, géographie et type de campagne            │
│                                                                         │
│  ┌──────────────────────────┬──────────────────────────┐               │
│  │ Performance par Device   │ Heatmap Jour × Heure     │               │
│  │ 🖥 Desktop  43%          │ (visuel HTML Content)    │               │
│  │   €265,4K / 4,0% CVR     │   Lun 1,5 2,0 12,0 27,6  │               │
│  │ 📱 Mobile   52%          │   Mar 1,1 1,7 13,7 26,4  │               │
│  │   €329,9K / 4,0% CVR     │   ... 7 lignes Lun-Dim    │               │
│  │ 📊 Tablet    5%          │ × 8 colonnes 0-3 → 21-24 │               │
│  │   €30,3K / 3,9% CVR      │                          │               │
│  └──────────────────────────┴──────────────────────────┘               │
│  ┌──────────────────────────┬──────────────────────────┐               │
│  │ Spend par Pays (vert)    │ Répartition Spend par    │               │
│  │ Côte d'Ivoire €248,9K    │ Campaign Type — stacked  │               │
│  │ Sénégal €123,8K          │ bar 2 années (2023/2024)│               │
│  │ Cameroun €93,2K          │ Display, PerfMax, Search,│               │
│  │ Burkina €64K, ...        │ Shopping, Video          │               │
│  └──────────────────────────┴──────────────────────────┘               │
└─────────────────────────────────────────────────────────────────────────┘
```

#### Visuel 1 — Performance par Device (3 cards horizontales compactes)

Chaque card = **icône colorée** (~30px gauche) + **label** + **valeur %** + sous-card en-dessous avec **Spend** + **mini-bar CVR**.

| Card | Icône | Couleur | Mesures |
|---|---|---|---|
| 🖥 **Desktop** | `monitor` | Violet `#7B61FF` | `[% Clicks Desktop]` → 43% · `CALCULATE([Total Spend], device="Desktop")` → €265,4K · CVR 4,0% |
| 📱 **Mobile** | `smartphone` | Bleu `#1B9CFC` | `[% Clicks Mobile]` → 52% · `CALCULATE([Total Spend], device="Mobile")` → €329,9K · CVR 4,0% |
| 📊 **Tablet** | `tablet` | Orange `#F39C12` | `[% Clicks Tablet]` → 5% · `CALCULATE([Total Spend], device="Tablet")` → €30,3K · CVR 3,9% |

> **🎓 ASTUCE — Mini-bars CVR violet en dessous**
>
> Pour reproduire les 3 mini-bars violettes : utiliser un **visuel "Histogramme empilé"** par device avec `[Conversion Rate]` en valeur. Couleur uniforme violet `#7B61FF`. Désactiver tous les axes (X et Y) et la légende. Hauteur fixe ~45 px.

#### Visuel 2 — Heatmap Jour × Heure (HTML)

```
Visuel : HTML Content (depuis AppSource)
└── Values
    └── [Heatmap Jour Heure HTML]   ← mesure créée au dossier 10
```

Le HTML est auto-contenu (CSS inline + fond transparent). Désactive : titre du visuel, fond du visuel, bordure (`Format → Général`).

> **🎓 MÉTHODE — Installer le visuel HTML Content**
>
> Insertion → Plus de visuels → **AppSource** → chercher *"HTML Content"* → choisir celui de **Daniel Marsh-Patrick** (gratuit, certifié) ou **HTML Content (lite)**.

#### Visuel 3 — Spend par Pays

```
Visuel : Histogramme à barres horizontales
├── Y → performance_quotidienne[country]
├── X → [Total Spend] / 1000
├── Tri : croissant par valeur
├── Couleur : vert uniforme #1D9E75
├── Format : €0,"K€" → €248,9K
└── Étiquettes : activées, à droite des barres
```

#### Visuel 4 — Répartition Spend par Campaign Type (stacked bar 2 années)

```
Visuel : Histogramme empilé (vertical)
├── Axe X      → dim_calendrier[Annee]
├── Axe Y      → [Total Spend]
├── Légende    → campaigns[campaign_type]
└── Couleurs   :
    ├── Display          : bleu vif #1B9CFC
    ├── Performance Max  : navy #1F3864
    ├── Search           : orange #EF9F27
    ├── Shopping         : violet #7B61FF
    └── Video            : bleu pâle #5DADE2
```

### 💡 INTERPRÉTATION — Lecture des 4 segments

> 📱 **Device** : Mobile-first confirmé (52%) mais Desktop solide (43%) — c'est cohérent avec le mix B2C+B2B du portefeuille agence. **Tablet à 5% : à ignorer**, ne pas optimiser dessus (règle des 10%).
>
> 🔥 **Horaire** : pic absolu **9-12h tous les jours ouvrés** (24-28 k€). Weekend -15% mais le créneau 9-12 reste fort. **Boost d'enchères 9-12 lun-jeu = +15-20%** sur ces tranches.
>
> 🌍 **Géographie** : la **Côte d'Ivoire concentre 40% du spend** (€248,9K / €625K). Sénégal en 2e à 20%. Les 4 marchés Afrique de l'Ouest sont équilibrés mais la CI reste le moteur.
>
> 🏷 **Campaign type 2023 vs 2024** : la stacked bar montre que **Search reste dominant** mais Performance Max gagne du terrain en 2024. C'est la tendance Google Ads attendue (PMax pousse depuis 2022).

### 🏢 MÉTIER — Les 3 optimisations Breakdown à présenter au CODIR

1. **Dayparting agressif nuit (0-6h)** : -50% d'enchères, économie estimée **€18-22K/an** sans impact volume.
2. **Boost 9-12 ouvré** : +20% d'enchères, gain estimé **+€35-45K de conversions value/an**.
3. **Tester PMax sur le Sénégal et Cameroun** : la stacked bar montre que ces marchés sont sous-représentés en PMax. Pilote 60 jours sur €5K budget.


In [ ]:
-- Requête DAX de validation Page 05

-- 1. Performance par Device
EVALUATE
SUMMARIZECOLUMNS(
    performance_quotidienne[device],
    "Pct Clicks", DIVIDE([Total Clicks], CALCULATE([Total Clicks], ALL(performance_quotidienne[device]))),
    "Spend", [Total Spend],
    "CVR", [Conversion Rate]
)
ORDER BY [Pct Clicks] DESC
-- Attendu : Mobile 52% €330K / Desktop 43% €265K / Tablet 5% €30K

-- 2. Heatmap : valeur d'une cellule (Lun 9-12)
EVALUATE
ROW(
    "Lun_9_12",
    CALCULATE(
        [Total Spend] / 1000,
        performance_quotidienne[day_of_week] = "Monday",
        FILTER(
            ALL(performance_quotidienne[hour]),
            performance_quotidienne[hour] >= 9 && performance_quotidienne[hour] < 12
        )
    )
)
-- Attendu : 27,6 (k€)

-- 3. Spend par Pays
EVALUATE
SUMMARIZECOLUMNS(
    performance_quotidienne[country],
    "Spend", [Total Spend]
)
ORDER BY [Spend] DESC
-- Attendu top 3 : Côte d'Ivoire €248,9K / Sénégal €123,8K / Cameroun €93,2K

---
## Étape 5 — Sous-titres dynamiques par page

### 🎓 MÉTHODE — Le pattern SELECTEDVALUE()

`SELECTEDVALUE(table[colonne], "valeur_par_defaut")` retourne :
- la valeur sélectionnée dans un slicer si **une seule** valeur est filtrée
- la valeur par défaut sinon (aucun filtre OU plusieurs valeurs)

C'est le pattern standard pour des **titres adaptatifs** qui suivent les filtres utilisateur.


In [ ]:
-- Sous-titres dynamiques pour les 5 pages
-- À créer comme mesures dans _Mesures (dossier "0. Sous-titres")
-- Puis insérer dans une zone de texte → "Insérer une valeur dynamique"

-- ════════════════════════════════════════════════════════
Sous Titre Page 01 =
"Vue globale des " &
COALESCE(SELECTEDVALUE(accounts[account_id]), "5 comptes") &
" · Performance " &
COALESCE(SELECTEDVALUE(dim_calendrier[Annee]) & "", "24 mois")

-- ════════════════════════════════════════════════════════
Sous Titre Page 02 =
[Nb Campagnes Actives] & " campagnes actives · Performance par ROAS"

-- ════════════════════════════════════════════════════════
Sous Titre Page 03 =
COUNTROWS(keywords) & " mots-clés · Performance et rétention dans le temps"

-- ════════════════════════════════════════════════════════
Sous Titre Page 04 =
FORMAT([Total Conversions], "#,##0") & " conversions · Répartition par type et coût unitaire"

-- ════════════════════════════════════════════════════════
Sous Titre Page 05 =
"Analyse par device, horaire, géographie et type de campagne"


---
## Étape 6 — Checklist de validation finale

### 🎓 MÉTHODE — Le test DAX vs Power BI

**Procédure :** ouvrir un éditeur de requêtes DAX (DAX Studio ou panneau Performance de Power BI) et comparer chaque KPI affiché avec la valeur attendue. Toute divergence = bug à corriger AVANT publication.

### ✅ Checklist 26 points

#### Mesures DAX clés (validation chiffres bruts)

| # | Mesure DAX | Valeur attendue | Statut |
|---|---|---|---|
| 1 | `[Total Impressions]` | **56 018 776** (~56,02M) | [ ] |
| 2 | `[Total Clicks]` | **1 488 092** (~1,49M) | [ ] |
| 3 | `[Total Conversions]` | **59 715** (~59,7K) | [ ] |
| 4 | `[CTR]` | **2,66%** | [ ] |
| 5 | `[CPC]` | **€0,42** | [ ] |
| 6 | `[Total Spend]` | **€625 488** (~€625,5K) | [ ] |
| 7 | `[Conversions Value]` | **€3,73M** | [ ] |
| 8 | `[ROAS]` | **5,96×** | [ ] |
| 9 | `[QS Moyen]` | **6,75** (affiché 6,7) | [ ] |
| 10 | `[Nb Keywords Actifs]` | **328** | [ ] |
| 11 | `[Keywords A Pauser]` | **23** | [ ] |
| 12 | `[Top CTR Keyword Valeur]` | **17,0%** | [ ] |
| 13 | `[Top CTR Keyword Nom]` | **brand hôtel sénégal p (Exact)** | [ ] |
| 14 | `[% Clicks Desktop]` | **43%** | [ ] |
| 15 | `[% Clicks Mobile]` | **52%** | [ ] |
| 16 | `[% Clicks Tablet]` | **5%** | [ ] |

#### Pages et navigation

| # | Élément à vérifier | Statut |
|---|---|---|
| 17 | Header présent sur les 5 pages avec logo + 5 onglets navigation | [ ] |
| 18 | Onglet actif souligné en violet `#7B61FF` | [ ] |
| 19 | Slicers de date (début/fin) en haut à droite, mêmes valeurs sur les 5 pages | [ ] |
| 20 | Sous-titres dynamiques se mettent à jour avec les slicers | [ ] |
| 21 | Page 02 : tableau utilise `campaigns[campaign_name]` (PAS `gads_campaigns_rank`) | [ ] |
| 22 | Page 03 : Cohort Retention triée par `mois_relatif` numérique (M0→M12 ordonné) | [ ] |
| 23 | Page 04 : Donut affiche les 6 types de conversions avec couleurs cohérentes | [ ] |
| 24 | Page 05 : Heatmap Jour Heure HTML s'affiche dans visuel HTML Content | [ ] |
| 25 | Theme couleur appliqué uniformément (`#7B61FF` / `#1D9E75` / `#E25F2F` / `#EF9F27`) | [ ] |
| 26 | **KPI cards Page 01** : 10 deltas affichés avec flèches ↑↓ et couleurs vert/rouge dynamiques | [ ] |


In [ ]:
-- ════════════════════════════════════════════════════════
-- REQUÊTE DAX FINALE DE VALIDATION GLOBALE
-- À exécuter une fois en fin de Power BI pour tout vérifier
-- ════════════════════════════════════════════════════════

-- Bloc 1 : KPIs Overview
EVALUATE
ROW(
    "Impressions",  [Total Impressions],
    "Clicks",       [Total Clicks],
    "Conversions",  [Total Conversions],
    "CTR",          [CTR],
    "CPC",          [CPC],
    "Spend",        [Total Spend],
    "CPM",          [CPM],
    "CostConv",     [Cost per Conversion],
    "ConvRate",     [Conversion Rate],
    "ConvValue",    [Conversions Value],
    "ROAS",         [ROAS]
)

-- Bloc 2 : Keywords & Devices
EVALUATE
ROW(
    "QS_Moyen",         [QS Moyen],
    "KW_Actifs",        [Nb Keywords Actifs],
    "KW_A_Pauser",      [Keywords A Pauser],
    "Top_CTR_Val",      [Top CTR Keyword Valeur],
    "Top_CTR_Nom",      [Top CTR Keyword Nom],
    "Pct_Desktop",      [% Clicks Desktop],
    "Pct_Mobile",       [% Clicks Mobile],
    "Pct_Tablet",       [% Clicks Tablet],
    "Pct_Total",        [% Clicks Desktop] + [% Clicks Mobile] + [% Clicks Tablet]
)
-- Vérif : Pct_Total = 1 (100%)

---
## Étape 7 — Storytelling CODIR (SCQRA 5 slides)

### 🎓 MÉTHODE — La structure SCQRA

Avant de présenter le dashboard à la direction de FluxData Agency, Marc-Aurèle doit raconter une **histoire claire** en 5 slides. Le framework **SCQRA** (Situation → Complication → Question → Réponse → Action) est utilisé par McKinsey et Barbara Minto pour structurer une narrative persuasive :

```
Slide 1 : Situation      — Où en sommes-nous aujourd'hui ?
Slide 2 : Complication   — Quel est le problème concret ?
Slide 3 : Question       — Quelle décision doit-on prendre ?
Slide 4 : Réponse        — Quelle est la recommandation chiffrée ?
Slide 5 : Action         — Quel est le plan d'exécution ?
```

### 📋 Slide 1 — SITUATION

**Titre :** *« FluxData Agency gère €625K de spend Google Ads / 24 mois pour 5 clients. »*

**Bullet points :**
- 5 comptes : TechShop CI · AfriHotels · BankAfrica · EdTechCI · MedSupply
- Volume total : 56,0M impressions · 1,49M clics · 59,7K conversions
- Performance : ROAS 5,96× (au-dessus du seuil 3× = rentable)
- Mix : 63% e-commerce (Purchase) · 28% B2B (Lead/Phone) · 9% SaaS (Signup)

**Visuel :** la page **01 Overview** du dashboard, screenshot.

### 📋 Slide 2 — COMPLICATION

**Titre :** *« Mais 4 heures de reporting manuel chaque lundi, et 5 campagnes drainent silencieusement le budget. »*

**Bullet points :**
- **4h/lundi × 4 semaines = 16h/mois** consacrées à compiler 5 fichiers Excel
- **5 campagnes en perte sèche** (ROAS < 1×) drainent **€72K** sur 24 mois (cf. Page 02 Bottom 5)
- **23 keywords à pauser** identifiés (QS ≤ 3, spend > €100) — gaspillage estimé **€8K/an**
- **0 visibilité sur la rétention keywords** par cohorte → impossible d'arbitrer ajout/retrait

**Visuel :** la page **02 Campaigns** Bottom 5 + extrait Page 03 Keywords.

### 📋 Slide 3 — QUESTION

**Titre :** *« Comment libérer 16h/mois de Marc-Aurèle ET récupérer €40K/an de gaspillage ? »*

**Visuel :** un schéma simple "AVANT / APRÈS" :
```
AVANT                          APRÈS
─────                          ─────
16h/mois reporting             1h/mois reporting (-94%)
0 visibilité campagnes         Top 5 / Bottom 5 auto
0 alerte gaspillage            23 keywords flagged
0 cohort retention             Suivi 13 mois M0-M12
0 dayparting                   Heatmap Jour × Heure
```

### 📋 Slide 4 — RÉPONSE

**Titre :** *« GoogleAdsPulse : un dashboard 5 pages déployé chez les 5 clients en self-service. »*

**Visuel :** mosaïque des 5 pages du dashboard (1 vignette par page : Overview / Campaigns / Keywords / Conversions / Breakdown).

**Bullet points :**
- **75 mesures DAX** prêtes à l'emploi (Volume, Efficacité, Rentabilité, Deltas MoM, Devices...)
- **Storytelling intégré** : chaque page répond à une question business précise
- **Self-service** : les clients peuvent filtrer eux-mêmes par compte, période, type de campagne
- **Coût marginal** : €5/mois infrastructure + €150/mois licences Power BI Pro × 3

### 📋 Slide 5 — ACTION

**Titre :** *« Plan 8 semaines — déploiement client par client, pilote puis généralisation. »*

**Planning :**

```
Semaines 1-2 : Développement modèle Power BI + 75 mesures DAX (FAIT via NB3)
Semaines 3-4 : Pilote sur TechShop CI — boucle de feedback utilisateur
Semaines 5-6 : Ajustements + déploiement AfriHotels + EdTechCI
Semaines 7-8 : Généralisation BankAfrica + MedSupply + formation équipe agence
```

**Ressources :**
- 1 Data Analyst à 50% (4 semaines)
- 1 Paid Media Manager à 20% (validation métier)
- Budget licences Power BI Pro : **€150/mois** × 3 utilisateurs

**Critère de succès à 3 mois :**
- Au moins **3 des 5 clients** utilisent le dashboard ≥ 1×/semaine en autonomie
- Marc-Aurèle gagne **≥ 16h/mois** sur le reporting
- **3 optimisations actées** parmi les 5 listées dans les pages Breakdown / Campaigns

**Visuel :** timeline Gantt 8 semaines + critères GO/NO-GO à chaque jalon.


---
## Étape 8 — Déploiement, RLS et refresh

### 8.1 — Fréquence de refresh

| Composant | Refresh | Mécanique |
|---|---|---|
| `performance_quotidienne` (CSV) | Quotidien 06h00 | Power BI Service — refresh planifié SharePoint/OneDrive |
| 7 CSV analytiques NB2 | Hebdomadaire lundi 05h00 | Script Python + DuckDB lancé par cron → upload SharePoint |
| Mesures DAX | Temps réel | Recalculées à chaque interaction utilisateur |

**Côté infrastructure :**
- CSV stockés sur **SharePoint entreprise FluxData**
- Power BI Service accède à SharePoint via OAuth
- Script Python tourne sur une VM Azure ~5 min/jour (coût ~€5/mois)

### 8.2 — Gestion des droits utilisateurs (Row-Level Security)

**3 niveaux d'accès :**

| Rôle | Accès | Utilisateurs |
|---|---|---|
| **Admin** | Édition + tous les comptes | Marc-Aurèle, Aïssatou (managers) |
| **Analyste Agence** | Lecture sur tous les comptes | Équipe data de l'agence |
| **Client** | Lecture SUR SON COMPTE uniquement | 5 clients (1 user principal par client) |

**Implémentation RLS :**

```dax
-- Dans Power BI Desktop : Modélisation → Gérer les rôles → Créer
-- Rôle "Client_TechShop"
-- Filtre sur accounts : [account_id] = "ACC001"

-- Rôle dynamique (recommandé) — basé sur l'email connecté
[account_owner_email] = USERPRINCIPALNAME()
```

Côté Power BI Service : **Sécurité du jeu de données** → assigner les utilisateurs aux rôles correspondants.

### 8.3 — Onboarding des 5 clients

| Semaine | Action | Livrable |
|---|---|---|
| S+1 | Démo individualisée 1h en visio | Replay enregistré |
| S+2 | Compte Power BI Pro créé + RLS activée | Lien dashboard envoyé |
| S+3 | Session Q&A 30 min de prise en main | FAQ partagée |
| S+4 | Premier feedback formel (NPS) | Score + verbatim |
| S+8 | Bilan mensuel d'usage (logs Power BI) | Rapport adoption |

### 🏢 MÉTIER — KPIs d'adoption à suivre

> Marc-Aurèle doit suivre **3 KPIs d'adoption** dans Power BI Service Audit Logs :
>
> 1. **Nb sessions / utilisateur / semaine** — cible : ≥ 1 par client
> 2. **Durée moyenne de session** — cible : ≥ 3 minutes (= vraie consultation, pas juste un clic)
> 3. **NPS interne** mesuré à 30 / 60 / 90 jours — cible : ≥ +30


---
## Bilan du Notebook 3

### Ce que tu as construit

| Livrable | Description |
|---|---|
| 📊 **Modèle en étoile** | 1 table de faits centrale + 5 dimensions + Calendrier custom + 6 tables analytiques NB2 |
| 🧮 **85 mesures DAX** | 11 dossiers organisés (Volume, Efficacité, Rentabilité, Deltas MoM, Conversions par type, Contextuelles, Keywords KPI, Devices, HTML Visuels, Helpers couleurs) |
| 📈 **5 pages thématiques** | Overview / Campaigns / Keywords / Conversions / Breakdown |
| 🎨 **Charte graphique** | Thème 5 couleurs cohérent (violet primaire, vert succès, rouge alerte, orange warning, bleu info) |
| 🔥 **Heatmap HTML** | Jour × Heure spend, fond transparent, palette 6 paliers |
| 🔁 **Cohort Retention** | Table calculée DAX pour analyse de rétention 13 mois |
| 🎯 **Storytelling SCQRA** | Narrative CODIR 5 slides pour FluxData Agency |
| 🚀 **Plan déploiement** | 8 semaines, RLS 3 niveaux, refresh quotidien automatisé |

### Le dashboard répond aux 5 questions clients

| Page | Question business | Réponse cible |
|---|---|---|
| 01 — Overview | Comment se comporte mon Paid Media ce mois-ci ? | 56M imp · 1,5M clics · €625K spend · ROAS 5,96× |
| 02 — Campaigns | Quelles campagnes performent et lesquelles drainent ? | Top : Brand-MedSupply 19,2× · Bottom : Search-MedicalEquip 0,3× |
| 03 — Keywords | Quels mots-clés pauser ? Booster ? | 23 à pauser · top CTR 17% · QS moyen 6,7 |
| 04 — Conversions | D'où viennent les conversions et combien coûtent-elles ? | 59,7K conv · 63% Purchase · Lead = CPA le moins cher (€7,53) |
| 05 — Breakdown | Quels segments optimiser ? | Mobile 52% · pic 9-12h · Côte d'Ivoire 40% du spend |

### 🏢 Les 5 priorités opérationnelles à présenter au CODIR

> 1. **🟢 Augmenter +30% les budgets Brand** (Top 5 ROAS = 11-19×) — gain estimé +€80K conversions value/mois
> 2. **🔴 Pauser 5 campagnes Bottom ROAS** (< 1×) — économie immédiate €72K sur 24 mois
> 3. **🟠 Pauser 23 keywords flagués** (QS≤3 + spend>€100) — gain net €8K/an
> 4. **🔥 Boost dayparting 9-12 ouvré** (+20% enchères) — gain estimé +€35-45K conv. value/an
> 5. **🌍 Tester PMax sur Sénégal et Cameroun** — pilote 60 jours, budget €5K

### Pour aller plus loin

- **🤖 Automatiser les alertes** : Power Automate déclenche un email à Marc-Aurèle quand une campagne passe en `[ROAS Verdict] = "❌ Non rentable"` plus de 3 jours consécutifs.
- **📱 Optimiser le mobile** : créer une **vue Téléphone** (Affichage → Disposition Téléphone) qui empile les visuels verticalement pour consultation en mobilité.
- **🧠 Aller plus loin avec ML** : entraîner un modèle de prédiction du ROAS J+30 par campagne (sur la base des 24 mois de données disponibles).


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.
